In [ ]:
import numpy as np
import astropy.io.fits as fits
from astropy.coordinates import SkyCoord, ICRS, Galactic
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.wcs import WCS
from matplotlib import cm
import reproject

In [ ]:
def make_plots_eq(data, wcs, RA_range=[24,0], dec_range=[0,90], 
                             vmin=-40, vmax=40, cmapuse = 'RdBu_r',cbarlbl='Jy/beam'):

    fs = 16
    crange = SkyCoord(RA_range, dec_range, frame=ICRS, unit=(u.hourangle, u.deg))

    fig = plt.figure(figsize=(18,6))
    ax = fig.add_subplot(111, projection=wcs.celestial)

    cmap = cm.get_cmap(cmapuse).copy()
    cmap.set_bad('lightgray')

    im = ax.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)

    ax.set_xlim(wcs.world_to_pixel(crange)[0])
    ax.set_ylim(wcs.world_to_pixel(crange)[1])

    cbar = fig.colorbar(im,ax=ax,orientation='vertical',fraction=0.01,pad=0.02,aspect=20)
    cbar.ax.tick_params(axis='y', which='both',labelsize=fs)
    cbar.set_label(cbarlbl, fontsize=fs)

    ax.tick_params(axis='both', which='major', labelsize=fs, zorder=30, length=5)
    ax.coords[0].display_minor_ticks(True)
    ax.coords[1].display_minor_ticks(True)
    ax.set_xlabel('RA',fontsize=fs)
    ax.set_ylabel('dec',fontsize=fs)

    #plt.tight_layout()
    #plt.savefig('/home/aordog/DVA_PLOTS/east_west_Jan03/'+prefix+'_'+freqname+'.pdf')
   
    return

In [ ]:
def feather_SA(image,dxy,R1=12.9,R2=17.1,*arg,**kwargs):

    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/610e6)*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/610e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = -0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 0
    mask[ruv < R1] = 1
    
    image_FFT_shift = image_FFT_shift*mask
    image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
       
    return image_SA, mask, uv_m, uv_n

In [ ]:
def feather_ST(image,dxy,R1=12.9,R2=17.1,*arg,**kwargs):

    plt.imshow(image)
        
    image_FFT = np.fft.fft2(image)

    print(np.nanmin(image_FFT.real))
    print(np.nanmax(image_FFT.real))

    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/512e6)*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/512e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = 0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 1
    mask[ruv < R1] = 0
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv_m, uv_n

In [ ]:
hdu  = fits.open('/srv/data/gmims/gmims-lbn/experiments/test7/dragons_test_Q.fits')
data = hdu[0].data
data[np.isnan(data)] = 0

print(data.shape)

dxy = np.round(hdu[0].header['CDELT2'],5)
print(dxy)

In [ ]:
make_plots_eq(data[0], WCS(hdu[0].header).dropaxis(2), RA_range=[24,0], dec_range=[-90,90], 
                             vmin=-2, vmax=2, cmapuse = 'RdBu_r',cbarlbl='Jy/beam')


In [ ]:
data_HP, mask_HP, uv_m, uv_n = feather_ST(data[0],dxy,R1=0.2,R2=0.4)

In [ ]:
print(np.nanmin(data_HP))

In [ ]:
make_plots_eq(data_HP.real, WCS(hdu[0].header).dropaxis(2), RA_range=[24,0], dec_range=[-90,90], 
                             vmin=-2, vmax=2, cmapuse = 'RdBu_r',cbarlbl='Jy/beam')

In [ ]:
def band_pass(image,dxy,freq,R1min=0.1,R1max=0.3,R2min=5.0,R2max=5.5,
              *arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)

    #print(np.nanmin(image_FFT.real))
    #print(np.nanmax(image_FFT.real))

    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/(freq*1e6))*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/(freq*1e6))*180/(np.pi**2)
    print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    # high pass:
    b = np.pi/(R1max-R1min)
    r0 = (R1max+R1min)/2.
    mask1 = 0.5*np.sin(b*(ruv - r0))+0.5
    mask1[ruv < R1min] = 0
    mask1[ruv > R2min] = 0
    mask1[np.where((ruv > R1max) & (ruv < R2min))] = 1

    # low pass:
    b = np.pi/(R2max-R2min)
    r0 = (R2max+R2min)/2.
    mask2 = -0.5*np.sin(b*(ruv - r0))+0.5
    mask2[ruv > R2max] = 0
    mask2[ruv < R2min] = 0
    #mask2[np.where((ruv > R1max) & (ruv < R2min))] = 1
 
    mask = mask1 + mask2

    #mask[np.where((ruv > R1max) & (ruv < R2min))] = 1
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv_m, uv_n

In [ ]:
data_HP, mask_HP, uv_m, uv_n = band_pass(data[0],dxy,512)

In [ ]:
#fig,ax = plt.subplots(1,1,figsize=(20,16))
#ax.imshow(mask_HP,vmin=0,vmax=1)

In [ ]:
make_plots_eq(data[0], WCS(hdu[0].header).dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
                             vmin=-5, vmax=5, cmapuse = 'RdBu_r',cbarlbl='Jy/beam')

make_plots_eq(data_HP.real, WCS(hdu[0].header).dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
                             vmin=-5, vmax=5, cmapuse = 'RdBu_r',cbarlbl='Jy/beam')

In [ ]:
hdu  = fits.open('/srv/data/gmims/gmims-lbn/experiments/test7/dragons_test_Q.fits')
dragons = hdu[0].data
dragons[np.isnan(dragons)] = 0

hdu  = fits.open('/srv/data/gmims/gmims-lbn/experiments/test7/chime_test_Q.fits')
chime = hdu[0].data
chime[np.isnan(chime)] = 0

print(dragons.shape)
print(chime.shape)

dxy = np.round(hdu[0].header['CDELT2'],5)
print(dxy)

In [ ]:
wcs =  WCS(hdu[0].header)

freqs = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6

print(freqs)

In [ ]:
fidx = abs(freqs-610.5).argmin()
print(freqs[fidx])

chime_filt,   mask, uv_m, uv_n = band_pass(chime[fidx],dxy,freqs[fidx],  R1min=0.2,R1max=0.3,R2min=5.0,R2max=15)
dragons_filt, mask, uv_m, uv_n = band_pass(dragons[fidx],dxy,freqs[fidx],R1min=0.2,R1max=0.3,R2min=5.0,R2max=15)

In [ ]:
vmax = 1

make_plots_eq(chime[fidx],       wcs.dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
              vmin=-vmax, vmax=vmax, cmapuse='RdBu_r',cbarlbl='Jy/beam')

make_plots_eq(chime_filt.real,   wcs.dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
              vmin=-vmax, vmax=vmax, cmapuse='RdBu_r',cbarlbl='Jy/beam')

make_plots_eq(dragons_filt.real, wcs.dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
              vmin=-vmax, vmax=vmax, cmapuse='RdBu_r',cbarlbl='K')

make_plots_eq(dragons[fidx],     wcs.dropaxis(2), RA_range=[24,0], dec_range=[0,90], 
              vmin=-vmax, vmax=vmax, cmapuse='RdBu_r',cbarlbl='K')
